# THzFit User Workflow

This notebook is for measured-data fitting in Google Colab or locally. It uploads or selects a measured reference and a measured sample trace, previews the raw data, previews the processed data used for fitting, then runs the fit for an arbitrary isotropic multilayer stack.


## Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Podrimate/THz_sim_application/blob/main/notebooks/THzFit_User_Workflow.ipynb)

Inside Colab, use the upload helpers below to choose the reference and sample CSV files directly from your computer. The notebook then preprocesses those traces and runs the fit.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PACKAGE_IMPORT_OK = False
DEFAULT_PIP_SPEC = 'git+https://github.com/Podrimate/THz_sim_application.git'

try:
    import thzsim2  # noqa: F401
    PACKAGE_IMPORT_OK = True
except Exception:
    PACKAGE_IMPORT_OK = False

if not PACKAGE_IMPORT_OK:
    repo_candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in repo_candidates:
        if (candidate / 'thzsim2').exists():
            sys.path.insert(0, str(candidate))
            try:
                import thzsim2  # noqa: F401
                PACKAGE_IMPORT_OK = True
                break
            except Exception:
                pass

if IN_COLAB and not PACKAGE_IMPORT_OK:
    pip_spec = os.environ.get('THZSIM_PIP_SPEC', DEFAULT_PIP_SPEC).strip()
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_spec])
    import thzsim2  # noqa: F401
    PACKAGE_IMPORT_OK = True

if not PACKAGE_IMPORT_OK:
    raise RuntimeError(
        'Could not import thzsim2. Open this notebook inside the repo, or in Colab let the install cell pull from GitHub.'
    )


## Setup / Imports

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from thzsim2.notebook_api import (
    ConstantNK,
    Drude,
    DrudeLorentz,
    Fit,
    Layer,
    Lorentz,
    LorentzOscillator,
    Measurement,
    NKFile,
    ReferenceStandard,
    plot_trace_pair_preview,
    prepare_trace_pair_for_fit,
    run_measured_fit,
)

workspace_root = Path.cwd()
if not (workspace_root / 'thzsim2').exists() and (workspace_root.parent / 'thzsim2').exists():
    workspace_root = workspace_root.parent

workflow_root = workspace_root / 'notebooks' / 'fit_workflow_outputs'
uploads_root = workflow_root / 'uploads'
workflow_root.mkdir(parents=True, exist_ok=True)
uploads_root.mkdir(parents=True, exist_ok=True)

print('workspace_root =', workspace_root)
print('workflow_root  =', workflow_root)


## Helper Functions

In [ ]:
def upload_single_csv(target_dir, *, prompt):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError('Colab upload is only available inside Google Colab.') from exc

    print(prompt)
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    name, payload = next(iter(uploaded.items()))
    path = target_dir / name
    path.write_bytes(payload)
    print('saved to', path)
    return path


def example_trace_paths(workspace_root, measurement_mode):
    root = Path(workspace_root) / 'Test_data_for_fitter'
    if measurement_mode == 'transmission':
        reference_csv = root / 'A11008858_transmission' / 'REFERENCE.csv'
        sample_csv = root / 'A11008858_transmission' / 'SAMPLE.csv'
    else:
        reference_csv = root / 'A11008858_reflection' / 'reflection_setup_ref_after_with_AuMirror_A1100858_avg600_onDryAir10min_int55.csv'
        sample_csv = root / 'A11008858_reflection' / 'reflection_setup_sample_A1100858_avg600_onDryAir10min_int27.csv'
    if reference_csv.exists() and sample_csv.exists():
        return reference_csv, sample_csv
    return None, None


def show_trace_metadata(prepared_traces):
    print('prepared metadata:')
    print(json.dumps(prepared_traces.metadata, indent=2))
    print('reference import metadata:')
    print(json.dumps(prepared_traces.raw_reference.metadata, indent=2))
    print('sample import metadata:')
    print(json.dumps(prepared_traces.raw_sample.metadata, indent=2))


def plot_measured_fit_summary(measured_fit_result):
    fit = measured_fit_result.fit_result
    prepared = measured_fit_result.prepared_traces
    fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=False)

    axes[0].plot(prepared.processed_reference.time_ps, prepared.processed_reference.trace, label='processed reference')
    axes[0].plot(prepared.processed_sample.time_ps, prepared.processed_sample.trace, label='processed sample')
    axes[0].plot(prepared.processed_sample.time_ps, fit['fitted_simulation']['sample_trace'], label='fit')
    axes[0].set_title('Measured Time-Domain Fit')
    axes[0].set_xlabel('Time (ps)')
    axes[0].set_ylabel('Signal')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(prepared.processed_sample.time_ps, fit['residual_trace'], label='residual')
    axes[1].set_title('Residual Trace')
    axes[1].set_xlabel('Time (ps)')
    axes[1].set_ylabel('Residual')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    omega = np.asarray(fit['fitted_simulation']['omega_rad_s'], dtype=np.float64)
    freq_thz = omega / (2.0 * np.pi * 1e12)
    mask = freq_thz >= 0.0
    response = np.asarray(fit['fitted_simulation']['sample_response'], dtype=np.complex128)
    axes[2].plot(freq_thz[mask], np.abs(response[mask]), label='|sample response|')
    axes[2].set_title('Optical Response Magnitude')
    axes[2].set_xlabel('Frequency (THz)')
    axes[2].set_ylabel('Magnitude')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    fig.tight_layout()
    print('recovered parameters:')
    print(json.dumps(fit['recovered_parameters'], indent=2))
    print('fitted measurement:')
    print(json.dumps(fit['fitted_measurement'], indent=2))
    return fig, axes


## 1. Choose The Reference And Sample Files

In Colab, upload one reference CSV and one sample CSV from your computer. Locally, you can point to paths or use the repository example data if it is available.

In [ ]:
measurement_mode = 'transmission'  # 'transmission' or 'reflection'
use_repo_example_data = not IN_COLAB
reference_csv_path = None
sample_csv_path = None


In [ ]:
example_reference_csv, example_sample_csv = example_trace_paths(workspace_root, measurement_mode)
if IN_COLAB:
    if reference_csv_path is None:
        reference_csv_path = upload_single_csv(uploads_root, prompt='Upload the measured reference CSV file.')
    if sample_csv_path is None:
        sample_csv_path = upload_single_csv(uploads_root, prompt='Upload the measured sample CSV file.')
elif use_repo_example_data and example_reference_csv is not None and example_sample_csv is not None:
    reference_csv_path = example_reference_csv
    sample_csv_path = example_sample_csv
else:
    if reference_csv_path is None or sample_csv_path is None:
        raise RuntimeError('Set reference_csv_path and sample_csv_path, or enable use_repo_example_data when local example files exist.')

print('reference_csv_path =', reference_csv_path)
print('sample_csv_path    =', sample_csv_path)


## 2. Preprocessing Settings

These settings define the exact traces that go into the fitter. The notebook will preview both the raw uploaded data and the processed data.

In [ ]:
reference_time_column = None
reference_signal_column = None
sample_time_column = None
sample_signal_column = None
baseline_subtract = True
baseline_window_samples = 40
crop_time_window_ps = (2620.0, 2740.0) if measurement_mode == 'transmission' else None


In [ ]:
prepared_traces = prepare_trace_pair_for_fit(
    reference_csv_path,
    sample_csv_path,
    reference_time_column=reference_time_column,
    reference_signal_column=reference_signal_column,
    sample_time_column=sample_time_column,
    sample_signal_column=sample_signal_column,
    baseline_subtract=baseline_subtract,
    baseline_window_samples=baseline_window_samples,
    crop_time_window_ps=crop_time_window_ps,
)
show_trace_metadata(prepared_traces)
preview_fig, preview_axes = plot_trace_pair_preview(prepared_traces)


## 3. Measurement Model

For v1, unknown polarization is modeled with one effective `s/p` mixing parameter. Set `angle_deg` and `polarization_mix` to either fixed numbers or `Fit(...)` if you want to fit them.

For reflection, the uploaded reference is treated as the physical reference standard, so this notebook uses `ReferenceStandard(kind='identity')` by default.

In [ ]:
measurement_config = Measurement(
    mode=measurement_mode,
    angle_deg=Fit(8.0, abs_min=0.0, abs_max=35.0, label='measurement_angle_deg'),
    polarization='mixed',
    polarization_mix=Fit(0.5, abs_min=0.0, abs_max=1.0, label='measurement_polarization_mix'),
    reference_standard=ReferenceStandard(kind='identity'),
)


## 4. Sample Structure And Fit Parameters

Edit this cell to define any isotropic multilayer stack supported by THzSim2. The example below is a one-layer Drude model seeded for the repository fitter example data, with thickness near `0.55 mm`.

In [ ]:
fit_layers = [
    Layer(
        name='drude_layer',
        thickness_um=Fit(550.0, abs_min=300.0, abs_max=800.0, label='film_thickness_um'),
        material=Drude(
            eps_inf=12.0,
            plasma_freq_thz=Fit(1.1, abs_min=0.1, abs_max=3.0, label='film_plasma_freq_thz'),
            gamma_thz=Fit(0.08, abs_min=0.005, abs_max=0.5, label='film_gamma_thz'),
        ),
    )
]


## 5. Fit Options

In [ ]:
fit_options = {
    'out_dir': workflow_root / 'fit_outputs' / f'{measurement_mode}_fit',
    'metric': 'mse',
    'optimizer': {
        'method': 'L-BFGS-B',
        'options': {'maxiter': 20},
        'global_options': {'maxiter': 2, 'popsize': 6, 'seed': 123},
        'fd_rel_step': 1e-4,
    },
    'max_internal_reflections': 4,
    'n_in': 1.0,
    'n_out': 1.0,
    'overlay_imported': True,
}


## 6. Run The Measured Fit

In [ ]:
measured_fit_result = run_measured_fit(
    prepared_traces,
    layers=fit_layers,
    out_dir=fit_options['out_dir'],
    measurement=measurement_config,
    optimizer=fit_options['optimizer'],
    metric=fit_options['metric'],
    max_internal_reflections=fit_options['max_internal_reflections'],
    n_in=fit_options['n_in'],
    n_out=fit_options['n_out'],
    overlay_imported=fit_options['overlay_imported'],
)

print('fit output dir =', measured_fit_result.out_dir)
print('artifacts =', measured_fit_result.artifact_paths)


## 7. Review The Fit

In [ ]:
fit_summary_fig, fit_summary_axes = plot_measured_fit_summary(measured_fit_result)
fit_summary_fig


## Colab Sharing

After this notebook is on GitHub, the public Colab link is:

`https://colab.research.google.com/github/Podrimate/THz_sim_application/blob/main/notebooks/THzFit_User_Workflow.ipynb`

Inside Colab, users only need to open the notebook, run the install cell, upload the reference and sample CSV files, and then edit the measurement/sample cells if needed.